# 🎬 AI Video Generator — Colab Render Engine
Zero-cost pipeline: **Edge-TTS** (voice) + **Pollinations.ai** (AI images) + **Pexels** (stock video) + **Pixabay** (music) + **FFmpeg** (render).

**Workflow:** `app.py` (Streamlit) generates `project_config.json` → upload it here → render → MP4 saved to Google Drive.

Run cells top to bottom. 🔽

## 1. Install dependencies & check GPU

In [ ]:
!pip -q install edge-tts requests
!apt-get -qq install -y ffmpeg >/dev/null
import subprocess, shutil
print('ffmpeg:', shutil.which('ffmpeg'))
has_gpu = shutil.which('nvidia-smi') is not None
print('GPU:', '✅ detected (will use h264_nvenc)' if has_gpu else '❌ none (CPU libx264)')
if has_gpu:
    print(subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True).stdout)

## 2. Mount Google Drive (output saves here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/AI_Videos'
import os; os.makedirs(OUT_DIR, exist_ok=True)
print('Output folder:', OUT_DIR)

## 3. Upload your `project_config.json` (from app.py)
Use the upload button that appears, OR place the file on your Drive.

In [ ]:
import os
CFG = '/content/project_config.json'
if not os.path.exists(CFG):
    from google.colab import files
    up = files.upload()
    if up:
        name = list(up.keys())[0]
        os.rename(name, CFG)
assert os.path.exists(CFG), 'project_config.json not found'
import json; c = json.load(open(CFG))
print('Loaded:', c.get('title','')[:60])
print(f"  format={c['format']} {c['resolution']['width']}x{c['resolution']['height']} fps={c.get('fps',30)}")
print(f"  scenes={len(c['scenes'])} voice={c['voice']} lang={c.get('language')}")

## 4. Enter your FREE API keys
- **Pexels** (free): https://www.pexels.com/api/ → sign up, get key
- **Pixabay** (free): https://pixabay.com/api/ → register, get key

Paste between the quotes.

In [ ]:
import os
os.environ['PEXELS_API_KEY']  = ''   # ← paste Pexels key
os.environ['PIXABAY_API_KEY'] = ''   # ← paste Pixabay key
assert os.environ['PEXELS_API_KEY'],  'Pexels key required (stock_video scenes)'
print('Keys set. Pexels len=%d Pixabay len=%d' % (
    len(os.environ['PEXELS_API_KEY']), len(os.environ['PIXABAY_API_KEY'])))

## 5. Fetch `render_script.py` and render 🚀
If you cloned the repo, it's already here. Otherwise we write it from the same folder.

In [ ]:
# Option A: it's already in the repo you cloned
RENDER = 'render_script.py'
if not os.path.exists(RENDER):
    # Option B: place render_script.py next to this notebook on Drive
    alt = os.path.join('/content/drive/MyDrive/AI_Videos', RENDER)
    if os.path.exists(alt):
        import shutil; shutil.copy(alt, RENDER)
assert os.path.exists(RENDER), 'render_script.py not found'

OUT_MP4 = os.path.join(OUT_DIR, (c.get('title') or 'video').lower().replace(' ','_')[:40] + '.mp4')
print('Rendering ->', OUT_MP4)

!python {RENDER} --config {CFG} --out "{OUT_MP4}" \
    --pexels-key "$PEXELS_API_KEY" --pixabay-key "$PIXABAY_API_KEY"

## 6. Preview & confirm

In [ ]:
import os
print('Exists:', os.path.exists(OUT_MP4))
print('Size: %.1f MB' % (os.path.getsize(OUT_MP4)/1e6)) if os.path.exists(OUT_MP4) else None
print('Saved to Google Drive:', OUT_MP4)
from google.colab import files
files.download(OUT_MP4)  # also download to your device